# Analyse des Réseaux Sociaux — Instagram
**Étude de la diffusion d'une plainte virale contre Royal Air Maroc (RAM)**

Ce notebook est auto-contenu : les données sont intégrées directement.

## 1. Installation des dépendances

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "uninstall", "community", "-y", "-q"])
subprocess.run([sys.executable, "-m", "pip", "install", "python-louvain", "-q"])

# Redémarre le kernel automatiquement pour recharger les modules
print("Redémarrage du kernel...")
import os
os.kill(os.getpid(), 9)

## 2. Import des bibliothèques

In [ ]:
import json
import re
import base64
from collections import Counter, defaultdict

import networkx as nx
import community as community_louvain
import matplotlib.pyplot as plt

POST_OWNER = "__POST__"

## 3. Chargement des données depuis GitHub

> Le fichier JSON est téléchargé directement depuis GitHub — aucun upload nécessaire.

In [ ]:
import requests, json

url = "https://raw.githubusercontent.com/zakaria1011/sna-instagram/main/data/comments_reels_2026-06-06_03-27-47-199.json"

response = requests.get(url)
response.raise_for_status()
data = response.json()
print(f"Données chargées : {len(data)} commentaires")

## 4. Construction du graphe

In [ ]:
def build_graph(data):
    G = nx.DiGraph()
    for c in data:
        u = c.get("ownerUsername", "")
        if u:
            G.add_node(u, type="user")
            if G.has_edge(u, POST_OWNER):
                G[u][POST_OWNER]["weight"] += 1
            else:
                G.add_edge(u, POST_OWNER, weight=1, type="comment")
    G.add_node(POST_OWNER, type="post")
    for c in data:
        dst = c.get("ownerUsername", "")
        for r in c.get("replies", []):
            src = r.get("ownerUsername", "")
            if src and dst and src != dst:
                if G.has_edge(src, dst):
                    G[src][dst]["weight"] += 1
                else:
                    G.add_edge(src, dst, weight=1, type="reply")
    all_comments = data + [r for c in data for r in c.get("replies", [])]
    for c in all_comments:
        src = c.get("ownerUsername", "")
        for mention in re.findall(r"@([\w.]+)", c.get("text", "")):
            if mention != src and G.has_node(mention):
                if G.has_edge(src, mention):
                    G[src][mention]["weight"] += 1
                else:
                    G.add_edge(src, mention, weight=1, type="mention")
    return G

G = build_graph(data)
print(f"Graphe : {G.number_of_nodes() - 1} nœuds, {G.number_of_edges()} arêtes")

## 5. Analyse

In [ ]:
def analyse(G):
    U = G.to_undirected()
    components = list(nx.connected_components(U))
    largest_nodes = max(components, key=len)
    largest = U.subgraph(largest_nodes).copy()
    degree_cent  = nx.degree_centrality(largest)
    between_cent = nx.betweenness_centrality(largest, normalized=True)
    in_deg       = dict(G.in_degree())
    out_deg      = dict(G.out_degree())
    partition    = community_louvain.best_partition(largest)
    communities  = Counter(partition.values())
    comm_members = defaultdict(list)
    for node, cid in partition.items():
        if node != POST_OWNER:
            comm_members[cid].append((node, degree_cent.get(node, 0)))
    return {
        "graph": G, "undirected": U, "largest": largest,
        "components": components, "degree_cent": degree_cent,
        "between_cent": between_cent, "in_deg": in_deg, "out_deg": out_deg,
        "partition": partition, "communities": communities,
        "comm_members": comm_members,
    }

results = analyse(G)
print("Analyse terminée.")

## 6. Rapport

In [ ]:
G2 = results["graph"]
largest    = results["largest"]
components = results["components"]
dc         = results["degree_cent"]
bc         = results["between_cent"]
in_d       = results["in_deg"]
out_d      = results["out_deg"]
communities  = results["communities"]
comm_members = results["comm_members"]

def top(d, n=10, exclude=POST_OWNER):
    return [(k, v) for k, v in sorted(d.items(), key=lambda x: -x[1]) if k != exclude][:n]

print("=" * 55)
print("  SOCIAL NETWORK ANALYSIS — INSTAGRAM COMMENTS")
print("=" * 55)
print(f"
Nœuds (utilisateurs)  : {G2.number_of_nodes() - 1}")
print(f"Arêtes (interactions) : {G2.number_of_edges()}")
print(f"Densité               : {nx.density(G2):.6f}")
print(f"Composantes connexes  : {len(components)}")
print(f"Plus grande composante: {len(max(components, key=len)) - 1} nœuds")
print("
--- TOP 10 DEGRÉ DE CENTRALITÉ ---")
for u, s in top(dc): print(f"  {u:<35} {s:.4f}")
print("
--- TOP 10 BETWEENNESS ---")
for u, s in top(bc): print(f"  {u:<35} {s:.4f}")
print("
--- TOP 10 IN-DEGREE ---")
for u, s in top({n: in_d.get(n, 0) for n in largest.nodes}): print(f"  {u:<35} {s}")
print("
--- TOP 10 OUT-DEGREE ---")
for u, s in top({n: out_d.get(n, 0) for n in largest.nodes}): print(f"  {u:<35} {s}")
print(f"
--- COMMUNAUTÉS ---")
print(f"  Nombre : {len(communities)}")
print(f"  Tailles : {sorted(communities.values(), reverse=True)[:15]}")
for cid, size in sorted(communities.items(), key=lambda x: -x[1])[:5]:
    members = [m[0] for m in sorted(comm_members[cid], key=lambda x: -x[1])[:5]]
    print(f"  Communauté {cid:>2} ({size:>4} membres): {members}")

## 7. Visualisation

In [ ]:
largest   = results["largest"]
partition = results["partition"]
dc        = results["degree_cent"]

interaction_nodes = [n for n in largest.nodes if largest.degree(n) > 1 or n == POST_OWNER]
sub = largest.subgraph(interaction_nodes).copy()

unique_comms = list(set(partition.get(n, -1) for n in sub.nodes))
cmap = plt.cm.get_cmap("tab20", len(unique_comms))
color_map = {c: cmap(i) for i, c in enumerate(unique_comms)}
node_colors = [color_map[partition.get(n, -1)] for n in sub.nodes]
node_sizes  = [max(50, dc.get(n, 0) * 5000) for n in sub.nodes]

fig, ax = plt.subplots(figsize=(16, 12))
pos = nx.spring_layout(sub, seed=42, k=0.4)
nx.draw_networkx_edges(sub, pos, alpha=0.2, edge_color="#aaaaaa", ax=ax)
nx.draw_networkx_nodes(sub, pos, node_color=node_colors, node_size=node_sizes, alpha=0.85, ax=ax)
labels = {n: n for n in sub.nodes if dc.get(n, 0) > 0.003}
nx.draw_networkx_labels(sub, pos, labels, font_size=7, ax=ax)
ax.set_title("Réseau d'interactions Instagram
(taille = centralité de degré, couleur = communauté)", fontsize=13)
ax.axis("off")
plt.tight_layout()
plt.show()